In [ ]:
# Data preparation function (batch processing)
def prepare_data(indices):
    # Batch construct neighbor index matrix
    full_indices = np.array([[i] + neighbors_indices[i].tolist() for i in indices], dtype=np.int32)
        
    # Batch extract feature data
    matrices = features_scaled[full_indices]  # (num_samples, 2196, 3)
        
    # Reshape dimensions
    matrices = matrices.reshape(-1, 13, 13, 13, 3)  # 2196 = 13^3
    matrices = np.transpose(matrices, (0, 4, 1, 2, 3))  # (num_samples, 3, 13, 13, 13)
    # Extract labels (only for training data)
    current_labels = labels[indices] if all(ZK[indices] == 1) else None
    
    return (
        torch.tensor(matrices, dtype=torch.float32),
        torch.tensor(current_labels, dtype=torch.long) if current_labels is not None else None,
        indices
    )

In [ ]:
# Data split
mask = (data['T'] == 1) & (data['YXML50'].isin([0, 1, 2]))
original_indices = np.where(mask)[0]  # get indices satisfying condition

train_indices, test_indices = train_test_split(
    original_indices, test_size=0.2, random_state=42, stratify=labels[original_indices]
)

# Prepare data loaders
train_matrices, train_labels, train_indexes = prepare_data(train_indices)
test_matrices, test_labels, test_indexes = prepare_data(test_indices)

In [ ]:
from torch.utils.data import DataLoader, Dataset

# Optimized RockDataset
import random

class RockDataset(Dataset):
    def __init__(self, matrices, labels, indexes, training=True):
        self.matrices = matrices
        self.labels = labels
        self.indexes = indexes
        self.training = training  # new training flag

    def __len__(self):
        return len(self.matrices)

        
    def __getitem__(self, idx):
        matrix = self.matrices[idx]
        label = self.labels[idx]
        index = self.indexes[idx]
        
        # Apply data augmentation only during training
        if self.training:
            matrix = self.random_drop(matrix)
            
        return matrix, label, index

    def random_drop(self, matrix, drop_prob=0.2, drop_ratio=0.2):
        """
        3D random dropout augmentation
        :param matrix: input tensor (C, D, H, W)
        :param drop_prob: probability of performing dropout
        :param drop_ratio: ratio of dropout region relative to the whole cube
        :return: augmented tensor
        """
        if random.random() > drop_prob:
            return matrix
        
        C, D, H, W = matrix.shape
        
        # Compute dropout region dimensions
        drop_d = int(D * drop_ratio)
        drop_h = int(H * drop_ratio)
        drop_w = int(W * drop_ratio)
        
        # Randomly select dropout location
        x = random.randint(0, D - drop_d)
        y = random.randint(0, H - drop_h)
        z = random.randint(0, W - drop_w)
        
        # Create mask and apply dropout
        mask = torch.ones_like(matrix)
        mask[:, x:x+drop_d, y:y+drop_h, z:z+drop_w] = 0
        return matrix * mask

In [ ]:
# Create data loaders
# Explicitly specify training parameter when creating dataset
train_dataset = RockDataset(train_matrices, train_labels, train_indexes, training=True)
test_dataset = RockDataset(test_matrices, test_labels, test_indexes, training=False) 
 
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [ ]:
# Compute class weights
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights (using 'balanced' mode)
# Convert PyTorch tensor to NumPy array
train_labels_np = train_labels.cpu().numpy()  # if on GPU, move to CPU first

# Check data type
if not np.issubdtype(train_labels_np.dtype, np.number):
    # If labels are string type, convert to numeric
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    train_labels_np = le.fit_transform(train_labels_np)
    classes = np.arange(len(le.classes_))  # update classes parameter
else:
    classes = np.unique(train_labels_np)


class_counts = np.array([np.sum(train_labels_np == c) for c in classes])

inv_counts = 1.0 / class_counts
normalized_weights = inv_counts / np.sum(inv_counts)

# Convert to PyTorch tensor
class_weights = torch.tensor(normalized_weights, dtype=torch.float32).to(device)

In [ ]:
# EarlyStopping class
class EarlyStopping:
    def __init__(self, patience=5,min_delta=0):
        self.patience = patience
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import balanced_accuracy_score


# Or use label-smoothing Focal Loss
class LabelSmoothingFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, smoothing=0.1, reduction='mean'):
        super(LabelSmoothingFocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing
        self.reduction = reduction
        
    def forward(self, inputs, targets):
        num_classes = inputs.size(-1)
        
        # Label smoothing
        log_preds = F.log_softmax(inputs, dim=-1)
        with torch.no_grad():
            smooth_targets = torch.zeros_like(log_preds)
            smooth_targets.fill_(self.smoothing / (num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        
        # Compute KL divergence
        kl_loss = F.kl_div(log_preds, smooth_targets, reduction='none').sum(-1)
        
        # Compute Focal weight
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        p = torch.exp(-ce_loss)
        focal_weight = (1 - p) ** self.gamma
        
        # Combine losses
        loss = focal_weight * kl_loss
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AnisotropicConv3d(nn.Module):
    """
    Anisotropic convolution module: performs depthwise convolution along X, Y, Z directions,
    then mixes channels via 1x1x1 convolution.
    :param in_channels: number of input channels
    :param out_channels: number of output channels
    :param kernel_size: triplet (kx, ky, kz), convolution kernel size in each direction
    :param stride: stride (default 1; if >1, stride is consistent in all directions for simplicity)
    :param padding: triplet (padx, pady, padz), padding in each direction
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super(AnisotropicConv3d, self).__init__()
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size, kernel_size)
        if isinstance(padding, int):
            padding = (padding, padding, padding)

        # Depthwise convolution along X direction (kx,1,1)
        self.conv_x = nn.Conv3d(
            in_channels, in_channels,
            kernel_size=(kernel_size[0], 1, 1),
            stride=stride,
            padding=(padding[0], 0, 0),
            groups=in_channels,
            bias=False
        )
        # Depthwise convolution along Y direction (1,ky,1)
        self.conv_y = nn.Conv3d(
            in_channels, in_channels,
            kernel_size=(1, kernel_size[1], 1),
            stride=stride,
            padding=(0, padding[1], 0),
            groups=in_channels,
            bias=False
        )
        # Depthwise convolution along Z direction (1,1,kz)
        self.conv_z = nn.Conv3d(
            in_channels, in_channels,
            kernel_size=(1, 1, kernel_size[2]),
            stride=stride,
            padding=(0, 0, padding[2]),
            groups=in_channels,
            bias=False
        )
        # Pointwise convolution (channel mixing)
        self.pointwise = nn.Conv3d(in_channels, out_channels, kernel_size=1, bias=True)

    def forward(self, x):
        x = self.conv_x(x)
        x = self.conv_y(x)
        x = self.conv_z(x)
        x = self.pointwise(x)
        return x

In [ ]:
class LeNet5_3D(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.5):
        super(LeNet5_3D, self).__init__()
        # Anisotropic convolutional layers
        self.conv1 = AnisotropicConv3d(in_channels=3, out_channels=16,kernel_size=(3, 5, 5),stride=1,padding=(1, 2, 2))
        self.bn1 = nn.BatchNorm3d(16)

        self.conv2 = AnisotropicConv3d(in_channels=16,out_channels=32,kernel_size=(3, 5, 5),stride=1,padding=(1, 2, 2))
        self.bn2 = nn.BatchNorm3d(32)

        self.conv3 = AnisotropicConv3d(in_channels=32,out_channels=64,kernel_size=(3, 5, 5),stride=1,padding=(1, 2, 2))
        self.bn3 = nn.BatchNorm3d(64)

        self.conv4 = AnisotropicConv3d(in_channels=64,out_channels=128,kernel_size=(3, 5, 5),stride=1,padding=(1, 2, 2))
        self.bn4 = nn.BatchNorm3d(128)

        self.conv5 = AnisotropicConv3d(in_channels=128,out_channels=256,kernel_size=(3, 5, 5),stride=1,padding=(1, 2, 2))
        self.bn5 = nn.BatchNorm3d(256)

        # Pooling layer
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2, padding=1)
        # Global average pooling
        self.global_avg_pool = nn.AdaptiveAvgPool3d(1)

        # Fully connected layers
        self.fc1 = nn.Linear(256, 512)
        self.bn8 = nn.BatchNorm1d(512)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = F.leaky_relu(self.bn1(self.conv1(x)), 0.01)
        x = F.leaky_relu(self.bn2(self.conv2(x)), 0.01)
        x = self.pool(x)

        x = F.leaky_relu(self.bn3(self.conv3(x)), 0.01)
        x = F.leaky_relu(self.bn4(self.conv4(x)), 0.01)
        x = self.pool(x)

        x = F.leaky_relu(self.bn5(self.conv5(x)), 0.01)
        x = self.pool(x)

        x = self.global_avg_pool(x)
        x = x.view(x.size(0), -1)
        x = F.leaky_relu(self.bn8(self.fc1(x)), 0.01)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Use higher gamma value and label smoothing
criterion = LabelSmoothingFocalLoss(
    alpha=class_weights, 
    gamma=2.0,  # increased gamma value
    smoothing=0.2,  # label smoothing
    reduction='mean'
)
# Initialize model
num_classes = 3
model = LeNet5_3D(num_classes).to(device)
model = nn.DataParallel(model, device_ids=[0, 1, 2, 3, 4, 5, 6, 7])  # use multiple GPUs
learning_rate=1e-3 
#criterion = FocalLoss(alpha = class_weights, gamma=2)  # focus on hard examples

# Define optimizer with L2 regularization (weight decay)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

# Dynamic learning rate scheduler (adjusts learning rate based on validation loss)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=3, verbose=True
)

# Initialize early stopping
early_stopping = EarlyStopping(patience=30, min_delta=0.001)

In [ ]:
# Initialize loss and accuracy lists
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []
 
# Training loop
num_epochs = 200
best_test_loss = float('inf')  # initialize best test loss as infinity

for epoch in range(num_epochs):
    model.train()  # set to training mode
    running_loss = 0.0
    correct = 0
    total = 0
 
    for inputs, labels, indexes in train_loader:
        inputs, labels = inputs.to(device), labels.to(device).long()  # move inputs and labels to GPU
        optimizer.zero_grad()  # zero the gradients
        outputs = model(inputs)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

 
    # Compute average loss and accuracy on training set
    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100 * correct / total
 
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)
 
    print(f"Epoch [{epoch+1}/{num_epochs}], "
          f"Train Loss: {epoch_loss:.4f}, Train Accuracy: {epoch_accuracy:.2f}%")
 
    # --- Testing phase ---
    model.eval()  # set to evaluation mode
    test_loss = 0.0
    all_preds = []
    all_labels = []
 
    with torch.no_grad():  # clear gradients
        for inputs, labels, indexes in test_loader:
            inputs, labels = inputs.to(device), labels.to(device).long() # move inputs and labels to GPU
            # Use full precision (float32) during testing
            with torch.autocast(device_type='cuda', dtype=torch.float32):  
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
            test_loss += loss.item()
 
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
 
    # Compute test loss and accuracy
    test_loss /= len(test_loader)
    test_accuracy = balanced_accuracy_score(all_labels, all_preds) * 100  # use balanced accuracy
 
    # Append test loss and accuracy to lists
    test_losses.append(test_loss)
    test_accuracies.append(test_accuracy)
 
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")
 
    # Dynamically adjust learning rate (based on validation loss)
    scheduler.step(test_loss)

    # If current test loss is better than historical best, save the model
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        state_dict = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        torch.save(state_dict, '/root/autodl-tmp/best.pth')  # save the best model

    # Early stopping check
    early_stopping(test_loss)
    if early_stopping.early_stop:
        print(f"Early stopping at epoch {epoch+1}!")
        break

# Save the final model
torch.save(model.state_dict(), 'best.pth')

In [ ]:
# Create combined figure
plt.figure(figsize=(10, 5))

# Plot loss curves (left)
plt.subplot(1, 2, 1)
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Train Loss', color='blue')
plt.plot(range(1, len(test_losses) + 1), test_losses, label='Test Loss', color='red')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curves')
plt.legend()
plt.grid(True)

# Plot accuracy curves (right)
plt.subplot(1, 2, 2)
plt.plot(range(1, len(train_accuracies) + 1), train_accuracies, label='Train Accuracy', color='green')
plt.plot(range(1, len(test_accuracies) + 1), test_accuracies, label='Test Accuracy', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy Curves')
plt.legend()
plt.grid(True)

In [ ]:
# Adjust layout and save
plt.tight_layout()
save_path = "autodl-tmp/combined_curves.png"
#plt.savefig(save_path, dpi=100, bbox_inches='tight')
#print(f"Combined curves saved to: {save_path}")
#plt.show()

import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc
# Compute accuracy on target data
target_accuracy = 100 * correct / total
print(f"Target Data Accuracy: {target_accuracy:.2f}%")

# Compute precision, recall, and F1 score
precision = precision_score(all_labels, all_preds, average='macro')
recall = recall_score(all_labels, all_preds, average='macro')
f1 = f1_score(all_labels, all_preds, average='macro')

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

In [ ]:
# Compute confusion matrix
conf_matrix = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(conf_matrix)

def plot_confusion_matrix(conf_matrix, class_names, normalize=False, title='Confusion Matrix'):
    plt.figure(figsize=(10, 8))
    plt.imshow(conf_matrix, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(title)
    plt.colorbar()
    # Set axis labels
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)
    # Add numeric labels
    thresh = conf_matrix.max() / 2.
    for i in range(conf_matrix.shape[0]):
        for j in range(conf_matrix.shape[1]):
            plt.text(j, i, format(conf_matrix[i, j], 'd'),
                     horizontalalignment="center",
                     color="white" if conf_matrix[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()


class_names = ['Class 0', 'Class 1', 'Class 2']
plot_confusion_matrix(conf_matrix, class_names, normalize=True)
save_path = "autodl-tmp/Confusion Matrix"

In [ ]:

data = pd.read_csv(r"s50_modify_data.dat")

# Extract coordinates and features
coordinates = data[["X", "Y", "Z"]].values
features = data[["den", "sus", "res"]].values

kd_tree = cKDTree(coordinates)  # build spatial index tree using cKDTree to accelerate nearest neighbor search
N = 2196  # set number of nearest neighbors
# Query the N nearest neighbors for each point (excluding itself)
neighbors_indices = [kd_tree.query(point, k=N + 1)[1][1:] for point in coordinates]

# Standardize data
scaler = StandardScaler() 
features_scaled = scaler.fit_transform(features)

# Modified data preparation function
def pred_prepare_data(indices):
    # Batch construct neighbor index matrix
    full_indices = np.array([[i] + neighbors_indices[i].tolist() for i in indices], dtype=np.int32)
    
    # Batch extract feature data
    matrices = features_scaled[full_indices]  # (num_samples, 2197, 3)
    
    # Reshape dimensions
    matrices = matrices.reshape(-1, 13, 13, 13, 3)  
    matrices = np.transpose(matrices, (0, 4, 1, 2, 3))  # (num_samples, 3, 13, 13, 13)
    
    return (torch.tensor(matrices, dtype=torch.float32), indices)

# Prepare prediction data (including all data)
pred_indices = np.arange(len(data))  # all data indices
pred_matrices, pred_indexes = pred_prepare_data(pred_indices)  # ignore returned labels

class RockDataset(Dataset):
    def __init__(self, matrices, indexes, training=True):
        self.matrices = matrices
        self.indexes = indexes
        self.training = training  # new training flag

    def __len__(self):
        return len(self.matrices)

    def __getitem__(self, idx):
        matrix = self.matrices[idx]
        index = self.indexes[idx]
        
        # Apply data augmentation only during training
        if self.training:
            matrix = self.random_drop(matrix)
            
        return matrix, index

# Create data loader
pred_dataset = RockDataset(pred_matrices, pred_indexes, training=False)
pred_loader = DataLoader(pred_dataset, batch_size=256, shuffle=False)


# prediction_sample
prediction_data = []  # use list to collect prediction data
prediction_df = pd.DataFrame(columns=["X","Y","Z","den","sus","res","YXML50","T"]+[f'class_{i}' for i in range(num_classes)])
file_path = f'prediction_002.csv'

# Load the saved model
model = LeNet5_3D(num_classes).to(device) 
model.load_state_dict(torch.load('/root/autodl-tmp/best.pth'))  
model.eval()

with torch.no_grad():
    for inputs, batch_indices in pred_loader:  # directly get batch_indices from iteration result
        inputs = inputs.to(device)  # move data to GPU
        outputs = model(inputs)
        probs = F.softmax(outputs, dim=1).cpu().numpy()  # get probabilities for each class
        _, predicted = torch.max(outputs.data, 1)
        batch_indices = batch_indices.numpy()

        # Get corresponding rows from data and reset index
        original_rows = data.iloc[batch_indices].reset_index(drop=True)
        # Build DataFrame containing predicted labels and probabilities
        batch_prediction_df = original_rows.copy()
        # Remove possible class_i columns
        for col in batch_prediction_df.columns:
            if col.startswith('class_') or col == "YXML50":
                batch_prediction_df.drop(columns=[col], inplace=True)
        # Compute index of the class with highest probability for each sample
        max_prob_indices = np.argmax(probs, axis=1)

        # Add predicted class index to DataFrame
        batch_prediction_df['predicted_class'] = max_prob_indices
        # Add prediction probabilities to DataFrame
        batch_prob_df = pd.DataFrame(probs, columns=[f'class_{i}' for i in range(num_classes)])
        batch_prediction_df = pd.concat([batch_prediction_df, batch_prob_df], axis=1)

        # Collect into list
        prediction_data.append(batch_prediction_df)
        # After loop, concatenate all data
prediction_df = pd.concat(prediction_data, ignore_index=True)
prediction_df.to_csv(file_path, index=False, encoding='utf-8-sig')